# Things to do 
- append all the labelled data from "../dataset/db_leonard.json", "../dataset/db_ananya.json", "../dataset/db_bryan.json", "../dataset/db_ryan.json" into "../dataset/db_labelled.json"

- append raw data from article.json, hackernews.json, reddit.json, twitter.json into temp.json

- filter out any entries from db_labelled.json in temp.json and save as raw_data.json

- split db_labelled.json into half (training.json, eval.json)

In [17]:
import json
import os
import random

DATA_DIR = "../dataset"
SAVE_DIR = "../final_dataset"

LABELLED_FILES = [
    "db_leonard.json", 
    "db_ananya.json", 
    "db_bryan.json", 
    "db_ryan.json"
]

RAW_FILES = [
    "article.json", 
    "hackernews.json", 
    "reddit.json", 
    "twitter.json"
]

def load_json(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    else:
        print(f"[WARN] File not found, skipping: {filepath}")
        return []

def save_json(data, filename):
    filepath = os.path.join(SAVE_DIR, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(data):,} records to {filename}")

In [18]:
print("Combining Labelled Data")
labelled_data = []
for file_name in LABELLED_FILES:
    filepath = os.path.join(DATA_DIR, file_name)
    data = load_json(filepath)
    labelled_data.extend(data)
    print(f"Loaded {len(data)} records from {file_name}")

save_json(labelled_data, "db_labelled.json")

Combining Labelled Data
Loaded 600 records from db_leonard.json
Loaded 600 records from db_ananya.json
Loaded 200 records from db_bryan.json
Loaded 600 records from db_ryan.json
Saved 2,000 records to db_labelled.json


In [19]:
print("\nCombining Raw Data")
raw_data_combined = []
for file_name in RAW_FILES:
    filepath = os.path.join(DATA_DIR, file_name)
    data = load_json(filepath)
    raw_data_combined.extend(data)
    print(f"Loaded {len(data)} records from {file_name}")

save_json(raw_data_combined, "temp.json")


Combining Raw Data
Loaded 1834 records from article.json
Loaded 11800 records from hackernews.json
Loaded 15633 records from reddit.json
Loaded 20178 records from twitter.json
Saved 49,445 records to temp.json


In [20]:
print("\nFiltering Deduplicates")
# Extract all unique IDs from the labelled dataset for lightning-fast lookups
labelled_ids = {post.get("ID") for post in labelled_data if post.get("ID")}

# Keep only the posts whose ID is NOT in the labelled_ids set
filtered_raw_data = [post for post in raw_data_combined if post.get("ID") not in labelled_ids]

print(f"Filtered out {len(raw_data_combined) - len(filtered_raw_data)} overlapping records.")
save_json(filtered_raw_data, "raw_data.json")


Filtering Deduplicates
Filtered out 2001 overlapping records.
Saved 47,444 records to raw_data.json


In [21]:
print("\nSplitting Train and Eval")
# Shuffle the data to ensure a balanced distribution of sources/sentiments
random.shuffle(labelled_data)

# Calculate the exact midpoint
midpoint = len(labelled_data) // 2

# Slice the lists
training_data = labelled_data[:midpoint]
eval_data = labelled_data[midpoint:]

save_json(training_data, "training.json")
save_json(eval_data, "eval.json")

print("\nAll tasks completed successfully!")


Splitting Train and Eval
Saved 1,000 records to training.json
Saved 1,000 records to eval.json

All tasks completed successfully!


In [ ]:
FILES_TO_CHECK = ["training.json", "eval.json", "raw_data.json"]

print("Sanity Check")

for file_name in FILES_TO_CHECK:
    filepath = os.path.join(SAVE_DIR, file_name)
    
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            print(f"{file_name} : {len(data):,} records")
    else:
        print(f"{file_name} : [ERROR] File not found!")


Sanity Check
training.json : 1,000 records
eval.json : 1,000 records
raw_data.json : 47,444 records
